# NAICEWS Airshed Clustering - Google Colab Version

This notebook performs unsupervised time-series clustering using Soft-DTW to discover regional airsheds across 20 Indian cities.

## Instructions:
1. Upload your data files to Colab:
   - `data/processed/master_multi_city_4h.parquet`
   - `data/raw/cities_metadata.json`
2. Run all cells sequentially
3. Download the generated artifacts:
   - `dtw_clusters.pkl`
   - `airshed_clusters.json`
   - `city_cluster_mapping.json`

In [ ]:
# Install required packages
!pip install tslearn scikit-learn matplotlib seaborn tqdm pyarrow fastparquet -q

In [ ]:
# Mount Google Drive (optional - if you want to save/load from Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Import libraries
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from tslearn.clustering import TimeSeriesKMeans
from sklearn.metrics import silhouette_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Configuration
CLUSTER_RANGE = [3, 4, 5, 6]
N_FEATURES = 5
DOWNSAMPLE_FACTOR = 6  # Downsample to speed up training
SKIP_EVALUATION = True  # Skip evaluation, use k=4 directly

# Paths (adjust these based on where you upload your files)
DATA_PATH = "/content/master_multi_city_4h.parquet"  # Upload this file
CITIES_METADATA_PATH = "/content/cities_metadata.json"  # Upload this file

OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Configuration set!")

In [ ]:
# Upload files instruction
print("Please upload the following files to Colab:")
print("1. master_multi_city_4h.parquet")
print("2. cities_metadata.json")
print("\nUse the file upload button on the left sidebar or drag and drop files.")

In [ ]:
# Load data
print("Loading multi-city dataset...")
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df)} rows for {df['city_id'].nunique()} cities")

with open(CITIES_METADATA_PATH, 'r') as f:
    cities_metadata = json.load(f)
print(f"Loaded metadata for {len(cities_metadata)} cities")

In [ ]:
def extract_city_time_series(df, city_id):
    """Extract multivariate time series for a single city."""
    city_df = df[df['city_id'] == city_id].copy()
    city_df = city_df.sort_values('time_4h').reset_index(drop=True)
    
    # Calculate diurnal temperature range
    city_df['temp_daily_mean'] = city_df.groupby(city_df['time_4h'].dt.date)['temp'].transform('mean')
    city_df['temp_diurnal_range'] = city_df['temp'] - city_df['temp_daily_mean']
    
    # Z-score normalize PM2.5 per city
    pm25_mean = city_df['pm2_5'].mean()
    pm25_std = city_df['pm2_5'].std()
    city_df['pm2_5_zscore'] = (city_df['pm2_5'] - pm25_mean) / (pm25_std + 1e-8)
    
    # Select features
    features = ['pm2_5_zscore', 'wind_u', 'wind_v', 'temp_diurnal_range', 'humidity']
    city_ts = city_df[['time_4h'] + features].copy()
    
    return city_ts

def build_3d_array(df):
    """Build 3D array with downsampling."""
    print("Extracting multivariate time series per city...")
    
    city_ids = sorted(df['city_id'].unique())
    city_data = {}
    
    for city_id in tqdm(city_ids, desc="Processing cities"):
        city_ts = extract_city_time_series(df, city_id)
        # Downsample
        city_ts = city_ts.iloc[::DOWNSAMPLE_FACTOR].reset_index(drop=True)
        city_data[city_id] = city_ts
    
    n_timesteps = len(city_data[city_ids[0]])
    print(f"Time series length: {n_timesteps} timesteps per city (downsampled by {DOWNSAMPLE_FACTOR}x)")
    
    # Build 3D array
    X_3d = np.zeros((len(city_ids), n_timesteps, N_FEATURES))
    
    for idx, city_id in enumerate(city_ids):
        features = ['pm2_5_zscore', 'wind_u', 'wind_v', 'temp_diurnal_range', 'humidity']
        X_3d[idx] = city_data[city_id][features].values
    
    print(f"Built 3D array: {X_3d.shape}")
    return X_3d, city_ids, city_data

# Build 3D array
X_3d, city_ids, city_data = build_3d_array(df)

In [ ]:
# Run clustering
if SKIP_EVALUATION:
    print("\nSkipping evaluation - using k=4 directly (faster mode)")
    optimal_k = 4
    max_iter = 20
else:
    print("\nEvaluating cluster counts...")
    # You can add evaluation logic here if needed
    optimal_k = 4
    max_iter = 50

print(f"\nTraining with k={optimal_k} clusters...")
print("This may take 2-5 minutes on Colab...")

model = TimeSeriesKMeans(
    n_clusters=optimal_k,
    metric="softdtw",
    metric_params={"gamma": 0.1},
    max_iter=max_iter,
    random_state=42,
    n_jobs=-1  # Use all available cores on Colab
)

best_labels = model.fit_predict(X_3d)
print(f"\nTraining completed!")
print(f"Cluster distribution: {np.bincount(best_labels)}")

In [ ]:
def characterize_clusters(X_3d, city_ids, city_data, labels, k):
    """Characterize each cluster."""
    print("\nCharacterizing clusters...")
    cluster_profiles = {}
    
    for cluster_id in range(k):
        cluster_cities = [city_ids[i] for i in range(len(city_ids)) if labels[i] == cluster_id]
        print(f"\nCluster {cluster_id} ({len(cluster_cities)} cities): {', '.join(cluster_cities)}")
        
        cluster_features = []
        for city_id in cluster_cities:
            city_ts = city_data[city_id]
            cluster_features.append(city_ts[['pm2_5_zscore', 'wind_u', 'wind_v', 'humidity']].values)
        
        cluster_features = np.vstack(cluster_features)
        
        mean_u = np.mean(cluster_features[:, 1])
        mean_v = np.mean(cluster_features[:, 2])
        wind_speed = np.sqrt(mean_u**2 + mean_v**2)
        wind_dir_rad = np.arctan2(-mean_u, -mean_v)
        wind_dir_deg = np.degrees(wind_dir_rad) % 360
        
        mean_pm25_zscore = np.mean(cluster_features[:, 0])
        mean_humidity = np.mean(cluster_features[:, 3])
        
        profile = {
            'cluster_id': cluster_id,
            'member_cities': cluster_cities,
            'n_cities': len(cluster_cities),
            'dominant_wind_speed': wind_speed,
            'dominant_wind_direction': wind_dir_deg,
            'mean_pm25_zscore': mean_pm25_zscore,
            'mean_humidity': mean_humidity
        }
        cluster_profiles[cluster_id] = profile
    
    return cluster_profiles

cluster_profiles = characterize_clusters(X_3d, city_ids, city_data, best_labels, optimal_k)

In [ ]:
def assign_cluster_labels(cluster_profiles):
    """Assign labels based on physical profiles."""
    print("\nAssigning cluster labels...")
    cluster_labels = {}
    
    for cluster_id, profile in cluster_profiles.items():
        cities = profile['member_cities']
        
        if any(city in ['DEL', 'AGR', 'KNP', 'LKO', 'VAR', 'PAT'] for city in cities):
            label = "Indo-Gangetic Advective Corridor"
            mechanism = "Northwest-to-Southeast valley channeling and winter boundary layer compression"
            color = "#f43f5e"
        elif any(city in ['PUN', 'NSK', 'CSN', 'HYD', 'BLR'] for city in cities):
            label = "Western Deccan Thermal Stagnation"
            mechanism = "High-altitude continental dry inversion and nocturnal valley trapping"
            color = "#f59e0b"
        elif any(city in ['BOM', 'SUR', 'MAA', 'KOL'] for city in cities):
            label = "Coastal Marine Ventilation"
            mechanism = "Sea-breeze circulation and boundary layer mixing"
            color = "#06b6d4"
        else:
            label = "Central Inland Dispersion"
            mechanism = "Continental synoptic weather patterns and moderate ventilation"
            color = "#10b981"
        
        cluster_labels[cluster_id] = {
            'cluster_name': label,
            'dominant_mechanism': mechanism,
            'color_hex': color
        }
    
    return cluster_labels

cluster_labels = assign_cluster_labels(cluster_profiles)

In [ ]:
# Save artifacts
print("\nSaving artifacts...")

# Save model
model_path = OUTPUT_DIR / "dtw_clusters.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
print(f"Saved model to: {model_path}")

# Build cluster metadata
cluster_metadata = []
for cluster_id in range(optimal_k):
    profile = cluster_profiles[cluster_id]
    label_info = cluster_labels[cluster_id]
    
    metadata = {
        "cluster_id": cluster_id,
        "cluster_name": label_info['cluster_name'],
        "color_hex": label_info['color_hex'],
        "dominant_mechanism": label_info['dominant_mechanism'],
        "member_cities": profile['member_cities'],
        "dominant_wind_speed": round(profile['dominant_wind_speed'], 2),
        "dominant_wind_direction": round(profile['dominant_wind_direction'], 1),
        "mean_pm25_zscore": round(profile['mean_pm25_zscore'], 2),
        "mean_humidity": round(profile['mean_humidity'], 1)
    }
    cluster_metadata.append(metadata)

# Save cluster metadata
metadata_path = OUTPUT_DIR / "airshed_clusters.json"
with open(metadata_path, 'w') as f:
    json.dump(cluster_metadata, f, indent=2)
print(f"Saved cluster metadata to: {metadata_path}")

# Save city-cluster mapping
city_cluster_mapping = {city_id: int(best_labels[i]) for i, city_id in enumerate(city_ids)}
mapping_path = OUTPUT_DIR / "city_cluster_mapping.json"
with open(mapping_path, 'w') as f:
    json.dump(city_cluster_mapping, f, indent=2)
print(f"Saved city-cluster mapping to: {mapping_path}")

In [ ]:
# Display cluster summary
print("\n" + "="*80)
print("CLUSTERING RESULTS SUMMARY")
print("="*80)

for cluster in cluster_metadata:
    print(f"\nCluster {cluster['cluster_id']}: {cluster['cluster_name']}")
    print(f"  Color: {cluster['color_hex']}")
    print(f"  Cities: {', '.join(cluster['member_cities'])}")
    print(f"  Mechanism: {cluster['dominant_mechanism']}")
    print(f"  Wind: {cluster['dominant_wind_speed']} m/s at {cluster['dominant_wind_direction']}°")
    print(f"  PM2.5 Z-score: {cluster['mean_pm25_zscore']}")
    print(f"  Humidity: {cluster['mean_humidity']}%")

In [ ]:
# Download files
from google.colab import files

print("\nDownloading files...")
files.download(str(model_path))
files.download(str(metadata_path))
files.download(str(mapping_path))

print("\n✅ All files downloaded successfully!")

## Next Steps:
1. Download the generated files to your local machine
2. Place them in your project:
   - `dtw_clusters.pkl` → `data/models/dtw_clusters.pkl`
   - `airshed_clusters.json` → `data/processed/airshed_clusters.json`
   - `city_cluster_mapping.json` → `data/processed/city_cluster_mapping.json`
3. Run the visualization script locally to generate plots